# Section 1: Gaussian Aperture Profiles

This notebook visualizes the input field profiles at the aperture plane before any diffraction computation. The truncation coefficient α controls how much of the aperture is illuminated by the Gaussian beam.

**Key concept:** The amplitude profile at the aperture is:

$$A(\rho) = \exp\!\left(-\alpha \rho^2\right), \quad \rho \in [0, 1]$$

where ρ = r/r_aperture is the normalized radial coordinate and α is the truncation coefficient.

- **α → 0**: uniform illumination (flat-top)
- **α = 1, 2**: moderately truncated Gaussian
- **α = 4**: nearly untruncated Gaussian (beam 1/e² radius ≈ aperture radius / 2)
- **α → ∞**: very narrow Gaussian, nearly all energy near center

For **annular apertures** with obscuration ratio ε, only the annular region ε ≤ ρ ≤ 1 transmits light. The amplitude across this region still follows the same Gaussian profile sampled from the same physical beam.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({'font.size': 12, 'figure.dpi': 100})

## 1.1 Gaussian Amplitude Profile A(ρ) for Different Truncation Coefficients

We define the normalized radial coordinate ρ = r/r_aperture ∈ [0, 1], where ρ = 1 is the aperture edge.
The intensity profile is I(ρ) = A(ρ)² = exp(−2α ρ²).

In [ ]:
# Normalized radial coordinate at the aperture plane
rho = np.linspace(0, 1.2, 500)  # extend slightly past aperture edge

# Truncation coefficients to compare
alphas = [1, 2, 4]
colors = ['tab:blue', 'tab:orange', 'tab:green']
labels = [f'α = {a}' for a in alphas]

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# --- Amplitude profiles ---
ax = axes[0]
for alpha, color, label in zip(alphas, colors, labels):
    A = np.exp(-alpha * rho**2)
    ax.plot(rho, A, color=color, lw=2, label=label)

ax.axvline(x=1.0, color='black', lw=1.5, ls='--', label='Aperture edge (ρ=1)')
ax.fill_betweenx([0, 1.05], 1.0, 1.2, alpha=0.12, color='gray', label='Blocked region')
ax.set_xlabel('Normalized radial coordinate ρ = r/r$_{ap}$')
ax.set_ylabel('Amplitude A(ρ)')
ax.set_title('Amplitude Profiles at Aperture Plane')
ax.set_xlim(0, 1.2)
ax.set_ylim(0, 1.05)
ax.legend()

# --- Intensity profiles ---
ax = axes[1]
for alpha, color, label in zip(alphas, colors, labels):
    I = np.exp(-2 * alpha * rho**2)
    ax.plot(rho, I, color=color, lw=2, label=label)

ax.axvline(x=1.0, color='black', lw=1.5, ls='--', label='Aperture edge (ρ=1)')
ax.fill_betweenx([0, 1.05], 1.0, 1.2, alpha=0.12, color='gray', label='Blocked region')
ax.set_xlabel('Normalized radial coordinate ρ = r/r$_{ap}$')
ax.set_ylabel('Intensity I(ρ) = A(ρ)²')
ax.set_title('Intensity Profiles at Aperture Plane')
ax.set_xlim(0, 1.2)
ax.set_ylim(0, 1.05)
ax.legend()

fig.suptitle('Gaussian Input Field Profiles at the Aperture Plane', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

# Print intensity at aperture edge for each α
print("Intensity at aperture edge (ρ=1):")
for alpha in alphas:
    I_edge = np.exp(-2 * alpha)
    print(f"  α = {alpha}: I(ρ=1) = exp(-{2*alpha}) = {I_edge:.4f} ({I_edge*100:.1f}% of peak)")

## 1.2 Truncation Concept: Beam Width vs Aperture

The truncation coefficient α relates the incident beam waist w_incident to the aperture radius r_ap:

$$\alpha = \left(\frac{r_{ap}}{w_{incident}}\right)^2$$

A larger α means the beam is narrower relative to the aperture — more of the beam energy passes through.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

rho_full = np.linspace(-1.2, 1.2, 800)

# Show full Gaussian beams (symmetric about axis) with aperture mask
for alpha, color, label in zip(alphas, colors, labels):
    A = np.exp(-alpha * rho_full**2)
    # Shade transmitted portion
    mask = np.abs(rho_full) <= 1.0
    ax.plot(rho_full, A, color=color, lw=1.5, ls='--', alpha=0.4)
    ax.plot(rho_full[mask], A[mask], color=color, lw=2.5, label=label)

# Mark aperture edges
ax.axvline(x=-1.0, color='black', lw=2, ls='-')
ax.axvline(x=1.0, color='black', lw=2, ls='-', label='Aperture edges')
ax.fill_betweenx([0, 1.05], -1.2, -1.0, alpha=0.15, color='gray')
ax.fill_betweenx([0, 1.05], 1.0, 1.2, alpha=0.15, color='gray', label='Blocked (opaque)')

ax.annotate('', xy=(1.0, 0.55), xytext=(-1.0, 0.55),
            arrowprops=dict(arrowstyle='<->', color='black', lw=1.5))
ax.text(0, 0.58, 'Aperture diameter 2$r_{ap}$', ha='center', fontsize=11)

ax.set_xlabel('Normalized radial coordinate ρ')
ax.set_ylabel('Amplitude A(ρ)')
ax.set_title('Gaussian Amplitude Profiles: Transmitted (solid) vs Blocked (dashed)')
ax.set_xlim(-1.2, 1.2)
ax.set_ylim(0, 1.05)
ax.legend(loc='upper right')
plt.tight_layout()
plt.show()

# Fraction of total Gaussian beam power transmitted
print("\nFraction of total Gaussian beam power transmitted through aperture:")
for alpha in alphas:
    # Power through aperture vs total (infinite beam): 1 - exp(-2α)
    f_transmitted = 1 - np.exp(-2 * alpha)
    print(f"  α = {alpha}: {f_transmitted*100:.1f}% transmitted")

## 1.3 Annular Aperture Profiles (Babinet's Principle)

An **annular aperture** blocks the central disk of radius ε·r_ap (obscuration ratio ε) and transmits the outer ring ε ≤ ρ ≤ 1.

The field from an annular aperture is computed via **Babinet's principle**:

$$E_{annular} = E_{outer\ disk}(NA) - E_{inner\ disk}(\varepsilon \cdot NA)$$

Here we show the aperture transmission functions for ε = 0.5 (moderate obscuration) and ε = 0.99 (strong annular, nearly all energy in the outer ring).

In [ ]:
alpha_fixed = 2  # α = 2 for illustration
epsilons = [0.0, 0.5, 0.99]
eps_colors = ['tab:blue', 'tab:orange', 'tab:red']
eps_labels = ['ε = 0 (full disk)', 'ε = 0.5', 'ε = 0.99']

rho_pos = np.linspace(0, 1.2, 600)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# --- 1D profiles ---
ax = axes[0]
for eps, color, label in zip(epsilons, eps_colors, eps_labels):
    A = np.where((rho_pos >= eps) & (rho_pos <= 1.0), np.exp(-alpha_fixed * rho_pos**2), 0.0)
    ax.plot(rho_pos, A, color=color, lw=2.5, label=label)

ax.axvline(x=1.0, color='black', lw=1.5, ls='--', alpha=0.7)
ax.fill_betweenx([0, 1.05], 1.0, 1.2, alpha=0.12, color='gray')
ax.set_xlabel('ρ = r/r$_{ap}$')
ax.set_ylabel('Amplitude A(ρ) · T$_{annular}$(ρ)')
ax.set_title(f'Annular Aperture Profiles (α = {alpha_fixed})')
ax.set_xlim(0, 1.2)
ax.set_ylim(-0.02, 1.05)
ax.legend()

# --- 2D aperture visualizations ---
ax = axes[1]

N = 300
x2d = np.linspace(-1.2, 1.2, N)
X, Y = np.meshgrid(x2d, x2d)
RHO = np.sqrt(X**2 + Y**2)

# Show the ε=0.5 annular profile in 2D
eps_show = 0.5
I2d = np.where((RHO >= eps_show) & (RHO <= 1.0), np.exp(-2 * alpha_fixed * RHO**2), 0.0)
im = ax.imshow(I2d, extent=[-1.2, 1.2, -1.2, 1.2], origin='lower',
               cmap='hot', vmin=0)
circle_outer = plt.Circle((0, 0), 1.0, fill=False, color='cyan', lw=2, label='Outer edge (ρ=1)')
circle_inner = plt.Circle((0, 0), eps_show, fill=False, color='lime', lw=2,
                           label=f'Inner edge (ρ=ε={eps_show})')
ax.add_patch(circle_outer)
ax.add_patch(circle_inner)
plt.colorbar(im, ax=ax, label='Intensity')
ax.set_title(f'2D Aperture Profile: ε = {eps_show}, α = {alpha_fixed}')
ax.set_xlabel('ρ$_x$')
ax.set_ylabel('ρ$_y$')
ax.legend(loc='upper right', fontsize=9)

fig.suptitle('Annular Aperture Input Field Profiles', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

## 1.4 Annular Profile for ε = 0.99 (Near-Ring Aperture)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

eps_show = 0.99
I2d_ring = np.where((RHO >= eps_show) & (RHO <= 1.0), np.exp(-2 * alpha_fixed * RHO**2), 0.0)

# 1D profile
ax = axes[0]
for alpha, color, label in zip(alphas, colors, labels):
    A = np.where((rho_pos >= eps_show) & (rho_pos <= 1.0),
                 np.exp(-alpha * rho_pos**2), 0.0)
    ax.plot(rho_pos, A, color=color, lw=2.5, label=label)
ax.axvline(x=1.0, color='black', lw=1.5, ls='--', alpha=0.7, label='Outer edge')
ax.axvline(x=eps_show, color='gray', lw=1.5, ls=':', label=f'Inner edge ε={eps_show}')
ax.set_xlabel('ρ = r/r$_{ap}$')
ax.set_ylabel('Amplitude')
ax.set_title(f'1D Annular Profile: ε = {eps_show}')
ax.set_xlim(0.9, 1.15)
ax.legend()

# 2D image
ax = axes[1]
im = ax.imshow(I2d_ring, extent=[-1.2, 1.2, -1.2, 1.2], origin='lower', cmap='hot', vmin=0)
circle_outer = plt.Circle((0, 0), 1.0, fill=False, color='cyan', lw=2, label='Outer edge')
circle_inner = plt.Circle((0, 0), eps_show, fill=False, color='lime', lw=2,
                           label=f'Inner edge ε={eps_show}')
ax.add_patch(circle_outer)
ax.add_patch(circle_inner)
plt.colorbar(im, ax=ax, label='Intensity')
ax.set_title(f'2D Aperture Profile: ε = {eps_show}, α = {alpha_fixed}')
ax.set_xlabel('ρ$_x$')
ax.set_ylabel('ρ$_y$')
ax.legend(loc='upper right', fontsize=9)

fig.suptitle('Near-Ring Aperture Profile (ε = 0.99)', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

## 1.5 Summary: Transmitted Power vs α and ε

The fraction of total Gaussian beam power transmitted through an annular aperture is:

$$\eta = e^{-2\alpha\varepsilon^2} - e^{-2\alpha}$$

In [ ]:
alpha_range = np.linspace(0.1, 6, 200)
eps_list = [0.0, 0.5, 0.99]
eps_colors2 = ['tab:blue', 'tab:orange', 'tab:red']

fig, ax = plt.subplots(figsize=(9, 5))

for eps, color in zip(eps_list, eps_colors2):
    eta = np.exp(-2 * alpha_range * eps**2) - np.exp(-2 * alpha_range)
    ax.plot(alpha_range, eta * 100, color=color, lw=2,
            label=f'ε = {eps}')

ax.axvline(x=1, color='tab:blue', ls=':', alpha=0.5)
ax.axvline(x=2, color='tab:orange', ls=':', alpha=0.5)
ax.axvline(x=4, color='tab:green', ls=':', alpha=0.5)
for a_mark, a_label in zip([1, 2, 4], ['α=1', 'α=2', 'α=4']):
    ax.text(a_mark + 0.05, 2, a_label, fontsize=10, color='gray')

ax.set_xlabel('Truncation coefficient α')
ax.set_ylabel('Transmitted power η (% of full Gaussian beam power)')
ax.set_title('Power Transmitted Through Aperture vs Truncation Coefficient')
ax.legend()
ax.set_xlim(0, 6)
ax.set_ylim(0, 100)
plt.tight_layout()
plt.show()

print("\nTransmitted power table (η in %)")
print(f"{'α':>6} | {'ε=0 (full)':>12} | {'ε=0.5':>10} | {'ε=0.99':>10}")
print("-" * 50)
for a in [1, 2, 4]:
    vals = []
    for eps in [0.0, 0.5, 0.99]:
        eta = np.exp(-2 * a * eps**2) - np.exp(-2 * a)
        vals.append(eta * 100)
    print(f"{a:>6} | {vals[0]:>12.1f} | {vals[1]:>10.1f} | {vals[2]:>10.1f}")

## Summary

| Concept | Description |
|---|---|
| α (truncation coeff.) | Controls how narrowly the Gaussian beam illuminates the aperture. Larger α → more truncation at edge |
| ε (obscuration ratio) | Fraction of aperture radius blocked by central stop. ε=0: full disk; ε→1: thin ring |
| Babinet's principle | E_annular = E_outer − E_inner; used to compute annular aperture diffraction |
| Gaussian reference NA | For Babinet subtraction, both inner and outer integrals use the same physical Gaussian beam (referenced to outer NA) |

The next notebooks (02–05) compute the actual diffracted fields using these input profiles.